# Time Series Data

References:

[1] McKinney, Wes. *Python for data analysis.* " O'Reilly Media, Inc.", 2022.

[2] VanderPlas, Jake. *Python data science handbook: Essential tools for working with data. Second Edition* " O'Reilly Media, Inc.", 2023.

[3] Johansson, Robert, Robert Johansson, and Suresh John. *Numerical python.* Vol. 1. New York: Apress, 2019.

Time series data is an important form of structured data in many fields, and its significance is its ability to capture and analyze information over time.
It has various applications such as in forecasting and prediction, monitoring and control, resource allocation and planning, financial analysis, healthcare and medicine, and many more.

In this notebook, explore how we deal with time series data, first by looking at the core data structures from Python's `datetime` library, then introducing extended time series data functionalities using `pandas`.


In [ ]:
import numpy as np
import pandas as pd

## 1 Date and Time Data Types and Tools

The `datetime` library is a python standard library that includes data types for date and time data, as well as calendar-related functionality.
The primary modules of this class is the `datetime`, `time`, and `calendar` modules.

In [ ]:
from datetime import datetime

In [ ]:
now = datetime.now()
now

In [ ]:
now.year, now.month, now.day

`datetime` stores the date and time down to the microsecond. A `timedelta` represent the temporal different between two date time objects.

In [ ]:
delta = datetime(2011, 1, 7) - datetime(2008, 6, 24, 8, 15)
delta

In [ ]:
delta.days

In [ ]:
delta.seconds

You can add (or subtract) `timedelta` to a `datetime` object to yield a shifted object:

In [ ]:
from datetime import timedelta

In [ ]:
start = datetime(2011, 1, 7)
start + timedelta(12)

In [ ]:
start - 2 * timedelta(seconds=10)

### Conversion Between String and Datetime

A common way to represent time is through strings. We can convert `datetime` objects to strings via the `strftime` method. For a list of different string conversion formats, refer to the table below:

<img src="images/datetime-format.png" style="width: 65%;">

In [ ]:
start.strftime("%Y-%m-%d")

In [ ]:
start.strftime("%F")

`datetime` objects also have a number of locale-specific formatting options for systems in other countries or languages.

In [ ]:
str(start)

<img src="images/local-specific.png" style="width: 55%;">

In [ ]:
start.strftime("%d %b %Y")

You can use many of the same format code to convert strings to dates using `datetime.strptime`

In [ ]:
value = '2011-01-03'

datetime.strptime(value, "%Y-%m-%d")

In [ ]:
datetime.strptime(value, "%F")

In [ ]:
datestrs = ['7/5/2022', '8/6/2011']

[datetime.strptime(x, '%m/%d/%Y') for x in datestrs]

`pandas` is generally oriented towards working with arrays of dates, whether used as an axis index or a column. The `pandas.to_datetime` method parses different kinds of date representations. Standard date formats like ISO 8601 can be parsed quickly:

In [ ]:
datestrs = ["2011-07-06 12:00:00", "2011-08-06 00:00:00"]
pd.to_datetime(datestrs)

## 2 Time Series Basics

A basic kind of time series object in pandas is a `Series` indexed by timestamps.

In [ ]:
rng = np.random.default_rng(1337)
time_series = pd.Series(rng.integers(10, size=(1_000)),
                        index=pd.date_range('2022-01-01', periods=1_000))
time_series

Under the hood, `datetime` objects have been put in the `DatetimeIndex`.

In [ ]:
time_series.index

### Indexing, Selection, Subsetting

Time series behaves like any other `Series` that we have indexed

In [ ]:
time_series['2024-03-09']

In [ ]:
time_series['2024-03-09':'2024-06-12']

Slicing with `datetime` objects also works

In [ ]:
time_series[datetime(2024, 3, 9):datetime(2024, 6, 12)]

In [ ]:
start_date = datetime(2024, 3, 9)
time_series[start_date:start_date + timedelta(days=100)]

## 3 Date Ranges, Frequencies, and Shifting

Time series in pandas are assumed to be irregular. For many applications this is sufficient. However, it is sometimes desirable to work relative to a time series with fixed frequency, such as daily, monthly, or even 15 minutes. `pandas` has a full suite of standard time series frequencie.

### Generating Date Ranges

`pandas.date_range` generates a `DatetimeIndex` with a length according to a particular frequency.

In [ ]:
ts_index = pd.date_range('2024-01-01', '2024-12-31')
ts_index

You may also pass a start or end date and a corresponding number of periods to generate.

In [ ]:
ts_index = pd.date_range(start='2024-01-01', periods=20)
ts_index

In [ ]:
ts_index = pd.date_range(end='2024-06-12', periods=100)
ts_index

In [ ]:
ts_index = pd.date_range(start='2024-01-01', periods=20, freq='s')
ts_index

The start and end dates define strict boundaries for the generated date index. But if you wanted a date index containing the last business day of each month. You would pass the `BM` frequncy.

In [ ]:
ts_index = pd.date_range('2022-01-01', '2024-12-31', freq='BME')
ts_index

For other frequency setting you can look at the documentation for other [DateOffset settings](https://pandas.pydata.org/docs/user_guide/timeseries.html#dateoffset-objects).

`pandas.date_range` preserves the time of the start or end timestamp.

In [ ]:
pd.date_range('2024-03-09 12:30:33', periods=5)

Sometimes you want to normalize to midnight as convention.

In [ ]:
pd.date_range('2024-03-09 12:30:33', periods=5, normalize=True)

### Frequencies and Date Offsets

Frequencies in pandas are composed of a *base frequency* and a multiplier. Bases frequencies are typically referred to by a string alias, like `M` for monthly or `H` for hourly. For each base frequency, there is an object referred to as a *date offset*.

In [ ]:
from pandas.tseries.offsets import Hour, Minute

In [ ]:
hour = Hour()
hour

In [ ]:
four_hours = Hour(4)
four_hours

You can use this to create date range objects, but usually you'd just use a string alias like `H` or `4H`.

In [ ]:
pd.date_range('2024-03-06', '2024-03-08', freq=four_hours)

In [ ]:
pd.date_range('2024-03-06', '2024-03-08', freq='4h')

Some frequencies describe points in time that are not evenly spaced. For example, `ME` refers to the calendar month end while `BM` refers to the last business/weekday of the month. We call these offsets as **anchored offsets**. See [Anchored Offsets](https://pandas.pydata.org/docs/user_guide/timeseries.html#anchored-offsets) for a comprehensive list.

In [ ]:
pd.date_range('2024-01-01', '2024-12-31', freq='ME')

In [ ]:
pd.date_range('2024-01-01', '2024-12-31', freq='BME')

One useful offset frequency is the [`WeekOfMonth`](https://pandas.pydata.org/docs/reference/api/pandas.tseries.offsets.WeekOfMonth.html#pandas.tseries.offsets.WeekOfMonth)

In [ ]:
from pandas.tseries.offsets import WeekOfMonth

In [ ]:
third_friday_of_the_month = WeekOfMonth(week=2, weekday=4)

In [ ]:
monthly_dates = pd.date_range('2024-01-01', '2024-12-31', freq=third_friday_of_the_month)
monthly_dates

In [ ]:
monthly_dates = pd.date_range('2024-01-01', '2024-12-31', freq='WOM-3FRI')
monthly_dates

### Shifting (Leading and Lagging) Data

*Shifting* refers to moving the data backward and forward through time. Both Series and DataFrame have a `shift` method for doing naive shifts forward or backward leaving the index unmodified.

In [ ]:
rng = np.random.default_rng(1337)
ts = pd.Series(rng.integers(10, size=10),
               index=pd.date_range('2024-03-01', periods=10, freq='MS'))
ts

In [ ]:
ts.shift(1)

In [ ]:
ts.shift(-2)

A common use of shift is to compute consecutive percent changes in the time series or multiple time series as DataFrame columns.

In [ ]:
ts / ts.shift(1) - 1

## 4 Periods and Period Arithmetic

*Periods* represent time spans, like days, months quarters, or years. The `pandas.Period` class represents this data type, requiring a string or integer and a supported frequency.

In [ ]:
p = pd.Period('2021', freq='Y-DEC')
p

In this case, the period object represents the full time span from Jan 1, 2021 to December 31, 2021.

`Period` and `PeriodIndex` objects can be converted to another frequency with their `asfreq` method.

In [ ]:
p = pd.Period('2011', freq='Y-DEC')
p

<img src="images/period-conversion.png" style="width: 65%;">

In [ ]:
p.asfreq('M', how='start')

In [ ]:
p.asfreq('M', how='end')

In [ ]:
p = pd.Period('2011-06', 'M')
p

In [ ]:
p.asfreq('Y-JUN', how='start')

### Quarterly Period Frequencies

Quarterly data is important in accounting, finance, and other fields. Quarterly data is usually reported relative to the fiscal year end, typically the last calendar or business day of one of the 12 months of the year. The period `2012Q4` has a different meaning depending on the fiscal year end.

In [ ]:
p = pd.Period('2012Q4', freq='Q-JAN')
p

In the case wherein the fiscal year ending in January, `2012Q4` runs from November 2011 through January 2012.

In [ ]:
p.asfreq('D', how='start')

In [ ]:
p.asfreq('D', how='end')

Arithmetic with respect to a period will follow its current frequency.

In [ ]:
p

In [ ]:
p + 1

In [ ]:
p + 2

### Converting Timestamps to Periods (and vice-versa)

In [ ]:
rng = np.random.default_rng(1337)
dates = pd.date_range('2024-01-01', periods=3, freq='ME')
ts = pd.Series(rng.integers(10, size=3), index=dates)
ts

In [ ]:
ts.to_period()

In [ ]:
ts.to_period('Q-JAN')

Since periods refer to non-overlapping time spans, a timestamp can only belong to a single period for a given frequency.

In [ ]:
rng = np.random.default_rng(1337)
dates = pd.date_range('2024-01-01', periods=100)
ts = pd.Series(rng.integers(10, size=100), index=dates)
ts

In [ ]:
ts.to_period('M')

To convert to a timestamp, we use the `to_timestamp` method.

In [ ]:
ts.to_period('M').to_timestamp()

## 5 Resampling and Frequency Conversion

*Resampling* refers to the process of converting a time series from one frequency to another. Aggregating higher frequency data to lower frequency is called downsampling, while converting lower frequency to higher frequency is called upsampling.

`pandas` object are equipped with a `resample` method, which allows frequency conversion. It has a similar idioms with `groupby` through which you can `resample` then call an aggregation function.

In [ ]:
rng = np.random.default_rng(1337)
dates = pd.date_range('2000-01-01', periods=100)
ts = pd.Series(rng.integers(10, size=100), index=dates)
ts

In [ ]:
ts.resample('ME').mean()

In [ ]:
ts.resample('ME').mean()

### Downsampling

Downsampling is aggregating data to a lower frequency. The desired frequency define a bin edges taht are used to slice the time series into pieces to aggregate.

When downsampling, here are some thinks that you need to consider:

1. Which side of the interval is closed.
2. How to label each aggregated bin, either with the start of the interval or the end.

In [ ]:
dates = pd.date_range('2000-01-01', periods=12, freq='min')
ts = pd.Series(np.arange(1, len(dates) + 1), index=dates)
ts

In [ ]:
ts.resample('5min').sum()

In [ ]:
ts.resample('5min', closed='right').sum()

In [ ]:
ts.resample('5min', closed='right', label='right').sum()

#### Open-high-low-close (OHLC) resampling

In finance, a popular way to aggregate a time series is to compute the four values of each bucket: the first (open), last (close), maximum (high), and minimum (low) values.

In [ ]:
rng = np.random.default_rng(1337)
ts = pd.Series(rng.integers(10, size=len(dates)), index=dates)
ts

In [ ]:
ts.resample('5min').ohlc()

### Upsampling and Interpolation

Upsampling is converting from a lower frequency to a higher frequency, where no aggregation is needed.

In [ ]:
rng = np.random.default_rng(1337)
dates = pd.date_range('2000-01-01', periods=2, freq='W-WED')
data = pd.DataFrame(rng.integers(10, size=(2, 4)), index=dates,
                    columns=['Makati', 'Quezon City', 'Pasig', 'Manila'])
data

We use `asfreq` to convert to the higher frequency without any aggregation.

In [ ]:
data.resample('D').asfreq()

If you want to fill the `nans` you can use `fill` methods or use interpolation methods such as `fillna`.

In [ ]:
data.resample('D').ffill()

In [ ]:
data.resample('D').bfill()

In [ ]:
data.resample('D').ffill(2)

## 6 Moving Window Functions

**Moving window functions** are mathematical operations applied to a subset of consecutive data points within a time-series. These functions play a crucial role in smoothing the data, identifying trends, and computing aggregated statistics.

In [ ]:
data = pd.read_csv('data/AAPL.csv', parse_dates=['Date'], index_col=0)
data

We introduce the `rolling` operator, which is similar to `resample` and `groupby`. It can be called on a `Series` or a `DataFrame` along with a window, expressed as a number of periods.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from matplotlib import rcParams


rcParams.update({'figure.figsize': (10, 6), 'axes.spines.top': False,
                 'axes.spines.right': False, 'axes.labelsize': 14,
                 'axes.titlesize': 16, 'axes.titleweight': 'bold',
                 'figure.titlesize': 18, 'figure.titleweight': 'bold',
                 'lines.linewidth': 1})

In [ ]:
ax = data.Close.plot(label='Close')
data.rolling(15).mean().Close.plot(ax=ax, label='MA 15')

ax.legend(loc='upper left', bbox_to_anchor=(1., 1.))
ax.set_xlim(['2010', '2020'])
ax.set_ylabel("Stock Price (USD)")
ax.set_title("AAPL")

In [ ]:
ax = data.Close.plot(label='Close')
data.rolling(15).mean().Close.plot(ax=ax, label='MA 15')
data.rolling(50).mean().Close.plot(ax=ax, label='MA 50')

ax.legend(loc='upper left', bbox_to_anchor=(1., 1.))
ax.set_xlim(['2010', '2020'])
ax.set_ylabel("Stock Price (USD)")
ax.set_title("AAPL")

Check the [`pandas.DataFrame.rolling`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html#pandas-dataframe-rolling) documentation for other parameters of the rolling operaiton.

### Exponentially Weighted Function

An alternative to using fixed window size is to use a constant *decay factor* to give more weight to more recent observations. This is implemented through the [`pandas.DataFrame.ewm`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.ewm.html) method.

In [ ]:
ax = data.Close.plot(label='Close')
data.rolling(15).mean().Close.plot(ax=ax, label='Moving Average')
data.ewm(span=15).mean().Close.plot(ax=ax, label='Exponentially Smoothing')

ax.legend(loc='upper left', bbox_to_anchor=(1., 1.))
ax.set_xlim(['2018', '2020'])
ax.set_ylabel("Stock Price (USD)")
ax.set_title("AAPL")

### Binary Moving Window Functions

Some statistical operators like correlation and covariance operate on two timeseries.

In [ ]:
aapl = pd.read_csv('data/AAPL.csv', parse_dates=['Date'], index_col=0)
msft = pd.read_csv('data/MSFT.csv', parse_dates=['Date'], index_col=0)

In [ ]:
returns_aapl = aapl.pct_change()
returns_msft = msft.pct_change()

In [ ]:
ax = aapl.Close.plot(label='AAPL')
msft.Close.plot(ax=ax, label='MSFT')
ax.legend()

In [ ]:
returns_aapl.rolling(125).Close.corr(returns_msft.Close).plot();